# Lab 5: Writing Custom Evaluators

## Difficulty: Intermediate | ~45 min | Requires Lab 3

Learn the three categories of evaluators in LangSmith — heuristic checks, LLM-as-judge, and custom functions — and attach them to runs using the `evaluate()` API.

In [ ]:
!pip install -qU "langsmith>=0.1.0" "langchain-core>=0.2.0" "langchain-openai>=0.1.0" "openai>=1.0.0" "python-dotenv>=1.0.0" "pydantic>=2.0.0"

## Cell 2: Load Environment and Initialize Clients

Loads API keys and sets up the LangSmith client (for dataset access and evaluation) and the OpenAI client (for the LLM-as-judge evaluator).

In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY"

ls_client = Client()
judge_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
print("Clients ready")

## Cell 3: Load the Lab 3 Dataset

Pulls the `product-reviews` dataset from Lab 3. Each example has a raw review as input and a structured `ProductReview` output (product, rating, sentiment).

In [ ]:
dataset = ls_client.read_dataset(dataset_name="product-reviews")
examples = list(ls_client.list_examples(dataset_id=dataset.id))
print(f"Loaded {len(examples)} examples from '{dataset.name}'")
for ex in examples[:3]:
    print(f"  Input: {ex.inputs}")
    print(f"  Output: {ex.outputs}\n")

## Cell 4: Define the Target Function

This is the function we want to evaluate. It takes a review, runs the structured-output agent, and returns the parsed result as a dict. We'll run this on every dataset example and compare its output against the expected output using our three evaluators.

In [ ]:
from langchain_openai import ChatOpenAI

class ProductReview(BaseModel):
    product: str = Field(description="The exact product being reviewed")
    rating: int = Field(description="The star rating, from 1 to 5")
    sentiment: str = Field(description="positive, negative, or neutral")

model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)
structured_model = model.with_structured_output(ProductReview)

def target(inputs: dict) -> dict:
    parsed = structured_model.invoke(inputs["review_text"])
    return parsed.model_dump()

## Cell 5: Evaluator 1 — Schema Validation

A heuristic evaluator that checks whether the agent's output parses against the expected Pydantic schema. It returns a boolean: `True` if the output matches the schema, `False` otherwise. This is the simplest type of evaluator — a pure rule-based check with no LLM calls.

In [ ]:
from langsmith.evaluation import evaluate

def schema_validator(run, example) -> dict:
    """Check if the output parses against the ProductReview schema."""
    output = run.outputs
    try:
        ProductReview(**output)
        return {"key": "schema_valid", "score": True}
    except ValidationError as e:
        return {"key": "schema_valid", "score": False, "comment": str(e)}

print("Schema validator defined — checks Pydantic parsing on every output")

## Cell 6: Evaluator 2 — LLM-as-Judge

An LLM-as-judge evaluator that asks a separate model to score the helpfulness of the agent's output on a 1–5 scale. This is useful for subjective quality metrics like relevance, clarity, or helpfulness that can't be captured by rule-based checks.

In [ ]:
def helpfulness_judge(run, example) -> dict:
    """Ask an LLM to score output helpfulness from 1-5."""
    review = example.inputs["review_text"]
    output = run.outputs
    reference = example.outputs

    prompt = f"""Score the following extraction on a 1-5 helpfulness scale.
Review: {review}
Extracted: {output}
Reference: {reference}
Return ONLY a JSON object: {{"score": <int>, "reason": "<brief explanation>"}}"""

    response = judge_client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    import json
    result = json.loads(response.choices[0].message.content)
    return {"key": "helpfulness", "score": result["score"], "comment": result["reason"]}

print("Helpfulness judge defined — LLM scores output quality from 1-5")

## Cell 7: Evaluator 3 — Guardrail Compliance

A custom evaluator that checks whether the agent's output respects guardrail policies. Here we define two policies: sentiment must be one of the allowed values, and the rating must be between 1 and 5. This pattern is directly applicable to any policy-enforcement scenario.

In [ ]:
ALLOWED_SENTIMENTS = {"positive", "negative", "neutral"}

def guardrail_checker(run, example) -> dict:
    """Check that the output respects configured guardrail policies."""
    output = run.outputs
    violations = []

    if output.get("sentiment") not in ALLOWED_SENTIMENTS:
        violations.append(f"Invalid sentiment: {output.get('sentiment')}")
    if not (1 <= output.get("rating", 0) <= 5):
        violations.append(f"Rating out of range: {output.get('rating')}")
    if not output.get("product"):
        violations.append("Missing product name")

    return {
        "key": "guardrail_pass",
        "score": len(violations) == 0,
        "comment": "; ".join(violations) if violations else "All guardrails passed"
    }

print("Guardrail checker defined — enforces sentiment, rating, and product policies")

## Cell 8: Run the Evaluation

Attach all three evaluators to an `evaluate()` run against the Lab 3 dataset. The `evaluate()` API runs the target function on every example and passes each run + example pair to every evaluator, collecting scores in one pass.

In [ ]:
results = evaluate(
    target,
    data="product-reviews",
    evaluators=[schema_validator, helpfulness_judge, guardrail_checker],
    experiment_prefix="lab5-custom-evaluators",
)
print(f"Evaluation complete: {len(results._results)} examples scored")

## Cell 9: Summarize Results

Aggregate the per-example scores into a summary table. This gives you a quick view of how the agent performs across all three evaluation dimensions.

In [ ]:
all_results = results._results
schema_scores = [r["evaluation_results"]["results"][0].score for r in all_results]
helpfulness_scores = [r["evaluation_results"]["results"][1].score for r in all_results]
guardrail_scores = [r["evaluation_results"]["results"][2].score for r in all_results]

print("=== Evaluation Summary ===")
print(f"Schema Valid:    {sum(schema_scores)}/{len(schema_scores)} passed ({sum(schema_scores)/len(schema_scores)*100:.0f}%)")
print(f"Guardrail Pass:  {sum(guardrail_scores)}/{len(guardrail_scores)} passed ({sum(guardrail_scores)/len(guardrail_scores)*100:.0f}%)")
print(f"Helpfulness Avg: {sum(helpfulness_scores)/len(helpfulness_scores):.1f}/5")

print("\n=== Per-Example Details ===")
for i, r in enumerate(all_results):
    evals = r["evaluation_results"]["results"]
    print(f"Example {i+1}: schema={evals[0].score}, helpfulness={evals[1].score}/5, guardrail={evals[2].score}")

## Optional Exercise

Add a fourth evaluator called `sentiment_accuracy` that checks whether the agent's predicted sentiment matches the reference sentiment from the dataset. It should return a boolean score (True if they match, False otherwise). Attach it to the evaluation run and compare its pass rate against the schema validator.